## Bert parameters 

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    # ==========================================
    # 1. CORE PATHS & IDENTIFIERS
    # ==========================================
    output_dir="./bert-fine-tuned",  # REQUIRED. Path where checkpoints and model outputs will be written.
    # overwrite_output_dir=True,  # Use True if you want to overwrite the output directory. False will safely prevent losing previous runs.

    # ==========================================
    # 2. CORE HYPERPARAMETERS (The Essentials)
    # ==========================================
    num_train_epochs=3,  # Total number of training epochs. BERT typically needs 2 to 5 epochs for fine-tuning.
    per_device_train_batch_size=16,  # Batch size per GPU/CPU for training. BERT-base usually fits 16-32 on a 16GB GPU.
    per_device_eval_batch_size=16,  # Batch size for evaluation. Can usually be equal to or larger than train batch size.
    learning_rate=5e-5,  # Initial learning rate. BERT fine-tuning typically uses 2e-5, 3e-5, or 5e-5.
    weight_decay=0.01,  # Strength of weight decay (L2 regularization). Prevents overfitting. 0.01 is standard for BERT.
    warmup_ratio=0.1,  # Proportion of training to perform linear learning rate warmup (e.g., 10% of total steps). Helps stabilize early training.

    # ==========================================
    # 3. MEMORY OPTIMIZATION & SPEED
    # ==========================================
# 0.8776778656787654567887654567876545678987654567865456765456787654345676545676543456
# 0.8776778656787654567

    fp16=True,  # CRITICAL FOR MODERN GPUs. Enables Mixed Precision (FP16) training. Reduces VRAM usage by ~50% and speeds up training significantly.
    gradient_accumulation_steps=1,  # If you face Out-Of-Memory (OOM) errors, increase this (e.g., to 2 or 4) and decrease per_device_train_batch_size.
    gradient_checkpointing=False,  # Set to True to save massive VRAM at the cost of ~20% slower training. Use only if batch size 1 still causes OOM.
    dataloader_num_workers=4,  # Number of subprocesses for data loading. Set to number of CPU cores to prevent data bottlenecks.

    # ==========================================
    # 4. EVALUATION STRATEGY
    # ==========================================
    eval_strategy="epoch",  # When to evaluate. Options: "no", "steps", "epoch". "epoch" checks performance at the end of every epoch.
    # eval_steps=500,               # Only used if eval_strategy="steps". Evaluates every X steps.

    # ==========================================
    # 5. LOGGING & MONITORING
    # ==========================================
    logging_dir="./logs",  # Directory for storing TensorBoard logs.
    logging_strategy="steps",  # When to log training metrics.
    logging_steps=100,  # Logs loss/learning rate every X training steps to track real-time progress.
    report_to=["tensorboard"],  # Integration platforms. Can be ["wandb", "tensorboard", "none"].

    # ==========================================
    # 6. SAVING & CHECKPOINTING
    # ==========================================
    save_strategy="epoch",  # Must match eval_strategy if using load_best_model_at_end. Options: "no", "epoch", "steps".
    # save_steps=500,               # Only used if save_strategy="steps". Saves a checkpoint every X steps.
    save_total_limit=2,  # Maximum number of checkpoints to keep. Deletes older ones to save hard drive space.

    # ==========================================
    # 7. BEST MODEL SELECTION
    # ==========================================
    load_best_model_at_end=True,  # Automatically loads the best model found during evaluation at the end of training.
    metric_for_best_model="eval_loss",  # The metric to monitor for the "best" model. Can be "eval_loss", "accuracy", "f1", etc.
    greater_is_better=False,  # Set to False if monitoring loss (lower is better). Set to True for accuracy/f1 (higher is better).

    # ==========================================
    # 8. SEED & REPRODUCIBILITY
    # ==========================================
    seed=42,  # Random seed to ensure reproducibility across runs.
)

In [ ]:
# Modifying for Low-VRAM environments
per_device_train_batch_size=4,  # Lower the actual batch size per GPU step
gradient_accumulation_steps=4,  # Simulates an effective batch size of 16 (4 * 4) without taking VRAM
fp16=True,  # Cuts memory in half compared to FP32
gradient_checkpointing=True,  # Extreme memory saving technique if the above still crashesb

In [ ]:
# Tweaks for optimal model quality
learning_rate=2e-5,  # A slightly more conservative, stable learning rate for BERT
warmup_ratio=0.1,  # Allows the model to smoothly adjust to your custom data domain
weight_decay=0.01,  # Standard L2 regularization
load_best_model_at_end=True,  # Ensures you don't overfit on the last epoch

In [ ]:
# Tweaks for quick sanity checks
num_train_epochs=1,  # Just run 1 epoch to see if it finishes successfully
max_steps=20,  # Force stop after exactly 20 steps to check logging/saving paths
eval_strategy="no",  # Skip evaluation during initial debugging to save time
save_strategy="no",  # Don't waste space saving untuned weights

## BERT for Regression

In [ ]:
pip install transformers datasets scikit-learn accelerate

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ==========================================
# 0. APPLE SILICON DEVICE CHECK
# ==========================================
# This ensures PyTorch sees your MacBook's GPU via Metal
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Apple Silicon GPU (MPS) detected and ready!")
else:
    device = torch.device("cpu")
    print("⚠️ MPS not detected. Falling back to CPU.")

# ==========================================
# 1. LOAD THE DATASET
# ==========================================
print("Loading GLUE STSb dataset...")
raw_datasets = load_dataset("nyu-mll/glue", "stsb")

# ==========================================
# 2. INITIALIZE TOKENIZER & PREPROCESS
# ==========================================
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(
        examples["sentence1"], 
        examples["sentence2"], 
        truncation=True, 
        max_length=128
    )

print("Tokenizing datasets...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==========================================
# 3. DEFINE METRICS
# ==========================================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.squeeze(predictions)
    
    mse = mean_squared_error(labels, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(labels, predictions)
    
    return {"mse": mse, "rmse": rmse, "mae": mae}

# ==========================================
# 4. INITIALIZE THE MODEL 
# ==========================================
print("Initializing BERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=1
)
# The Trainer will automatically move the model to MPS during training, 
# but it's good practice to be aware of your device.

# ==========================================
# 5. MAC-OPTIMIZED TRAINING ARGUMENTS
# ==========================================
training_args = TrainingArguments(
    output_dir="./bert-mac-regression",
    
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # 🛑 CRITICAL MAC CHANGES HERE 🛑
    fp16=False,             # Disable FP16. It is optimized for NVIDIA CUDA and can cause kernel errors on Mac.
    bf16=False,             # Leave false unless you have an M3/M4 chip that explicitly supports bfloat16.
    use_cpu=False,          # Ensures the Trainer searches for a hardware accelerator (which will be MPS).
    
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none" 
)

# ==========================================
# 6. TRAIN & EVALUATE
# ==========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training on Mac GPU...")
trainer.train()

print("\nRunning final evaluation...")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

# ==========================================
# 7. TEST PREDICTION
# ==========================================
print("\nTesting out a single manual prediction...")
test_sentence_1 = "A man is playing a guitar."
test_sentence_2 = "A guy is strumming a guitar."

# Explicitly move inputs and model to the Mac GPU for custom inference
model.to(device)
inputs = tokenizer(test_sentence_1, test_sentence_2,
 return_tensors="pt", truncation=True).to(device)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    predicted_score = outputs.logits.item()

print(f"Sentence 1: '{test_sentence_1}'")
print(f"Sentence 2: '{test_sentence_2}'")
print(f"Predicted Similarity Score (0.0 to 5.0): {predicted_score:.2f}")

🚀 Apple Silicon GPU (MPS) detected and ready!
Loading GLUE STSb dataset...
Tokenizing datasets...
Initializing BERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14933.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Starting training on Mac GPU...


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Mse,Rmse,Mae
1,0.601707,0.525090,0.525090,0.724631,0.568353
2,0.389885,0.510541,0.510541,0.714522,0.535837
3,0.249800,0.490993,0.490993,0.700709,0.523081


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.70it/s]
/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.90it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.wei


Running final evaluation...


/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Mse,Rmse,Mae
0.249800,0.490993,3,0.490993,0.700709,0.523081


  eval_loss: 0.4910
  eval_mse: 0.4910
  eval_rmse: 0.7007
  eval_mae: 0.5231

Testing out a single manual prediction...
Sentence 1: 'A man is playing a guitar.'
Sentence 2: 'A guy is strumming a guitar.'
Predicted Similarity Score (0.0 to 5.0): 2.99
